In [14]:
%pip install tensorboard

  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------- 5.5/5.5 MB 48.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   ---------------------------------------- 4.7/4.7 MB 71.6 MB/s eta 0:00:00
Using cached tensorboard_data_server-0.7.2-py3-none-any.whl (2.4 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os 
from src.config import *
from src.tokenizer import Tokenizer

from src.model import create_seq2seq_model
from src.utils import count_parameters, load_checkpoint
from src.train import predict_corrections

import time


# Initialize tokenizer
tokenizer = Tokenizer(max_length=256)
tokenizer.load_vocab("vocab.json")

# Get vocab size from tokenizer
vocab_size = len(tokenizer.vocab)
print(f"Vocabulary size: {vocab_size}")

# Create model
print("Initializing model...")
model = create_seq2seq_model(
    vocab_size=vocab_size,
    hidden_dim=HIDDEN_DIM,
    embedding_dim=EMBEDDING_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    device='cpu'
)

print(f"Model has {count_parameters(model):,} trainable parameters")

# Initialize optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Specify checkpoint path (replace with actual path or set to None)
checkpoint_path = "models/best_model.pt"  # Set to None to start from scratch

# Load checkpoint if provided
if checkpoint_path is not None and os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    model, optimizer, metadata = load_checkpoint(model, optimizer, checkpoint_path, DEVICE)
    start_epoch = metadata['epoch'] + 1  # Resume from the next epoch
    best_val_loss = metadata['val_loss']
else:
    print("No checkpoint provided or found. Starting training from scratch.")
    start_epoch = 0
    best_val_loss = float('inf')


In [10]:
CHECKPOINT_PATH = r"C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\models\best_model.pt"

import torch
from src.config import *
from src.model import create_seq2seq_model
from src.tokenizer import Tokenizer
from src.utils import load_checkpoint

# Initialize tokenizer
tokenizer = Tokenizer(max_length=MAX_LENGTH)
tokenizer.load_vocab(TOKENIZER_VOCAB_PATH)

# Rebuild the same model used during training
vocab_size = len(tokenizer.vocab)
model = create_seq2seq_model(
    vocab_size=vocab_size,
    hidden_dim=HIDDEN_DIM,
    embedding_dim=EMBEDDING_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    n_heads=N_HEADS,
    device=DEVICE
)

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print(f"Loaded model from {CHECKPOINT_PATH}")


Loaded vocabulary with 356 tokens
Loaded model from C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\models\best_model.pt


In [15]:
import os 
from src.config import *
from src.tokenizer import Tokenizer

from src.model import create_seq2seq_model
from src.utils import count_parameters, load_checkpoint
from src.train import predict_corrections

import time
noisy_examples = [
    # "mən elmler metrosu yaxinliginda 5 mertebeli ev axtariram",
    "salam, aileli sexsler ucun 28 mayda metkebe yaxin erazide ev axtariram qiymeti 400-500 manat olsun.",
    # "qardaw oglu, mene xirdalandan ev lazimdi, genis ev isteyirem",
    # "dostlarimla qalmaq ucun ev axtariram, telebeyik",
    # "dostlarimla qalmaq ucun ev axtariram. telebeyik",
    # "eng klaviaturasi ile yazanlar vcun kifay't q'd'r yaxshidi m'nce"
    # "mene iki otqli kiraye bir ev lazmdlr.bu evn maks qiymeti 400 manat osun"
]

st= time.time()
corrected_examples = predict_corrections(
    model=model,
    tokenizer=tokenizer,
    texts=noisy_examples,
    device="cpu"
)
print(time.time() - st)


# Print results
for i, (noisy, corrected) in enumerate(zip( noisy_examples, corrected_examples)):
    print(f"\nExample {i+1}:")
    print(f"  Noisy:     {noisy}")
    print(f"  Corrected: {corrected}")

c:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\src\train.py:305: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\AZINTELECOM\anaconda3\Lib\site-packages\torch\amp\autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


0.5748004913330078

Example 1:
  Noisy:     salam, aileli sexsler ucun 28 mayda metkebe yaxin erazide ev axtariram qiymeti 400-500 manat olsun.
  Corrected: salamşaa le   a t                                                                                                              


In [17]:
paths = [
    r"C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\train\train_sentences_azeri.json",
    r"C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\train\train_words_azeri.json",
    r"C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\test\test_sentences_azeri.json",
    r"C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\train\test_words_azeri.json"    
]

import json

def load_json(path:str):
    with open(path, "r", encoding='utf-8') as fl:
        data = json.load(fl)
    return data

for x in paths:
    counter = 0
    data = load_json(x)
    for dp in data:
        cl, ns = dp.values()
        counter += len(find_number_patterns(cl))
    print(counter, x)
        

32970 C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\train\train_sentences_azeri.json
0 C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\train\train_words_azeri.json
8271 C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\test\test_sentences_azeri.json
0 C:\Users\AZINTELECOM\Downloads\seq2seq\seq2seq\data\train\test_words_azeri.json


In [16]:
import re
from typing import Dict, List

# Azerbaijani/Latin letters for suffixes and units
AZ_LETTERS = r"A-Za-zƏÖĞÇŞİÜəöğçşıü"

PATTERNS: Dict[str, re.Pattern] = {
    # 1) a  /  2) b   style enumerations
    "enum_simple": re.compile(
        rf"\b(\d+)\)\s*([{AZ_LETTERS}])"
    ),

    # 2.1-1, 3.2-4 kind of step ids
    "enum_complex": re.compile(
        r"\b\d+\.\d+-\d+\b"
    ),

    # plain number: 25, 2003, etc.
    "plain_number": re.compile(
        r"\b\d+\b"
    ),

    # number + single letter: 2c, 3c, 5b
    "number_letter": re.compile(
        rf"\b\d+[{AZ_LETTERS}]\b"
    ),

    # ordinal: 20-ci, 25-ci, 2003-cu etc.
    "ordinal": re.compile(
        r"\b\d+-(?:cı|ci|cu|cü|ncı|nci|ncu|ncü)\b",
        re.IGNORECASE,
    ),

    # numeric ranges: 20-25, 30-35
    "range": re.compile(
        r"\b\d+\s*-\s*\d+\b"
    ),

    # decimal with comma: 2,5 (correct)
    "decimal_comma": re.compile(
        r"\b\d+,\d+\b"
    ),

    # noisy decimal with double comma: 2,,5
    "decimal_double_comma": re.compile(
        r"\b\d+,,\d+\b"
    ),

    # fraction with slash: 2/5
    "fraction_slash": re.compile(
        r"\b\d+/\d+\b"
    ),

    # numeric suffixes like 100-luk, 10-luq, 5-lik etc.
    "numeric_suffix_luk": re.compile(
        r"\b\d+-(?:luk|lük|luq|lıq|lik|lik|liq)\b",
        re.IGNORECASE,
    ),

    # units with space: 5 kq, 6 kq, 10 mm
    "unit_spaced": re.compile(
        rf"\b\d+\s+(?:kq|kg|mm|sm|cm|m|l|ml)\b",
        re.IGNORECASE,
    ),

    # units attached: 5kq, 10mm, 3kg
    "unit_attached": re.compile(
        rf"\b\d+(?:kq|kg|mm|sm|cm|m|l|ml)\b",
        re.IGNORECASE,
    ),

    # percent with comma and suffix: 5,5%-li type
    "percent_decimal_suffix": re.compile(
        r"\b\d+,\d+%-li\b",
        re.IGNORECASE,
    ),

    # simple percent forms: 5%, 5 %-li
    "percent_simple": re.compile(
        r"\b\d+%(-li)?\b",
        re.IGNORECASE,
    ),
}


def find_number_patterns(text: str) -> Dict[str, List[str]]:
    """
    Find all number-related patterns in a string, grouped by type.
    Only checks the patterns described above.
    """
    results: Dict[str, List[str]] = {}
    for name, pattern in PATTERNS.items():
        matches = [m.group(0) for m in pattern.finditer(text)]
        if matches:
            results[name] = matches
    return results
